# Notebook 07 — Análisis de Imagen de Dron con rasterio
**Teledetección — Maestría en Ingeniería | Universidad del Magdalena**  
**Sesión 4 | Sábado 25 Jul 2026**

Este notebook analiza el ortomosaico multibanda del **DJI Phantom 4 Multispectral** que procesamos en WebODM esta mañana. Trabajaremos con Python puro — sin GEE — usando la librería `rasterio`.

### Lo que haremos
1. Cargar el ortomosaico de 5 bandas (producto de WebODM)
2. Visualizar cada banda individualmente
3. Calcular índices espectrales: NDVI, NDRE, NDMI
4. Hacer estadísticas básicas por zona (lago vs. vegetación vs. suelo)
5. Exportar los resultados como GeoTIFF

### Bandas del P4M (en orden en el stack)
| Banda | λ (nm) | Índice en array |
|-------|--------|----------------|
| Blue  | 450    | 0              |
| Green | 560    | 1              |
| Red   | 650    | 2              |
| Red Edge | 730 | 3             |
| NIR   | 840    | 4              |

---

## Parte 0 — Instalación

In [ ]:
# En entorno local (Conda): rasterio ya debe estar instalado
# En Colab:
try:
    import rasterio
    print(f'rasterio {rasterio.__version__} — OK')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'rasterio', '--quiet'])
    import rasterio

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from rasterio.plot import show, show_hist
from rasterio.transform import from_bounds
import rasterio.features
import warnings
warnings.filterwarnings('ignore')
print('Librerías listas')

## Parte 1 — Cargar el ortomosaico

El ortomosaico puede ser:
- Un solo archivo con 5 bandas apiladas: `ortomosaico_p4m_5bandas.tif` (creado con `gdal_merge.py`)
- Cinco archivos separados: `orthophoto_Blue.tif`, `orthophoto_Green.tif`, etc.

Este notebook maneja ambos casos.

In [ ]:
import os

# OPCIÓN A: archivo con 5 bandas apiladas (recomendado)
RUTA_ORTO = 'ortomosaico_p4m_5bandas.tif'

# Si estás en Colab, primero sube el archivo:
# from google.colab import files
# uploaded = files.upload()  # selecciona el .tif

# Verificar que el archivo existe
if not os.path.exists(RUTA_ORTO):
    print(f'⚠ Archivo no encontrado: {RUTA_ORTO}')
    print('Opciones:')
    print('  1. Sube el archivo desde WebODM (ortomosaico_p4m_5bandas.tif)')
    print('  2. Usa el dataset de ejemplo del bananal (se descarga en la celda siguiente)')
else:
    with rasterio.open(RUTA_ORTO) as src:
        print('=== Metadatos del ortomosaico P4M ===')
        print(f'Archivo:    {RUTA_ORTO}')
        print(f'Dimensiones:{src.width} × {src.height} px')
        print(f'Bandas:     {src.count}')
        print(f'CRS:        {src.crs}')
        print(f'Resolución: {src.res[0]*100:.1f} cm/px')
        print(f'Extensión:  {src.bounds}')
        print(f'Dtype:      {src.dtypes[0]}')

In [ ]:
# Si no tienes el archivo del vuelo de hoy, usa este dataset de ejemplo
# (imagen simulada con las mismas propiedades del P4M sobre el bananal)

def crear_imagen_ejemplo_p4m(ruta_salida='orto_ejemplo_p4m.tif', filas=800, cols=1200):
    """Genera un GeoTIFF multibanda sintético con propiedades del P4M."""
    np.random.seed(42)

    # Crear máscara de coberturas (en una imagen 800x1200)
    lago    = np.zeros((filas, cols), bool)
    cesped  = np.zeros((filas, cols), bool)
    arboles = np.zeros((filas, cols), bool)
    suelo   = np.zeros((filas, cols), bool)

    # Lago (zona superior izquierda)
    lago[50:250, 50:400] = True
    # Arboles (zona derecha)
    arboles[100:600, 700:1150] = True
    # Césped (zona central)
    cesped[300:700, 100:700] = True
    # Suelo (resto)
    suelo[~lago & ~arboles & ~cesped] = True

    # Reflectancias típicas P4M (escala 0-65535 para uint16)
    reflectancias = {
        'Blue': {'lago':800,'cesped':2200,'arboles':1800,'suelo':7000},
        'Green':{'lago':900,'cesped':3500,'arboles':2800,'suelo':8000},
        'Red':  {'lago':700,'cesped':2000,'arboles':1500,'suelo':7500},
        'RE':   {'lago':800,'cesped':5000,'arboles':9000,'suelo':7200},
        'NIR':  {'lago':600,'cesped':18000,'arboles':35000,'suelo':9000},
    }

    bandas = []
    for banda, vals in reflectancias.items():
        img = np.zeros((filas, cols), dtype=np.float32)
        ruido = 0.08  # 8% de variación aleatoria
        for cob, mascara in [('lago',lago),('cesped',cesped),('arboles',arboles),('suelo',suelo)]:
            v = vals[cob]
            img[mascara] = v * (1 + ruido * np.random.randn(np.sum(mascara)))
        bandas.append(img.astype(np.uint16))

    from rasterio.transform import from_origin
    transform = from_origin(-74.20, 11.00, 0.000001, 0.000001)

    with rasterio.open(
        ruta_salida, 'w', driver='GTiff',
        height=filas, width=cols, count=5,
        dtype=np.uint16, crs='EPSG:4326', transform=transform
    ) as dst:
        for i, banda in enumerate(bandas, 1):
            dst.write(banda, i)
            dst.update_tags(i, banda_nombre=['Blue','Green','Red','RedEdge','NIR'][i-1])

    print(f'Imagen de ejemplo creada: {ruta_salida} ({filas}×{cols} px, 5 bandas)')
    return ruta_salida

# Crear si no existe el archivo real
if not os.path.exists(RUTA_ORTO):
    RUTA_ORTO = crear_imagen_ejemplo_p4m()
    print('Usando imagen de ejemplo. Reemplaza con tu ortomosaico real de WebODM.')

## Parte 2 — Leer y visualizar las 5 bandas

In [ ]:
with rasterio.open(RUTA_ORTO) as src:
    blue  = src.read(1).astype(np.float32)
    green = src.read(2).astype(np.float32)
    red   = src.read(3).astype(np.float32)
    re    = src.read(4).astype(np.float32)  # Red Edge
    nir   = src.read(5).astype(np.float32)
    perfil = src.profile.copy()
    transform = src.transform
    crs = src.crs

# Normalizar a [0,1] para visualización
def normalizar(banda, percentil=2):
    p_low  = np.percentile(banda[banda > 0], percentil)
    p_high = np.percentile(banda[banda > 0], 100 - percentil)
    return np.clip((banda - p_low) / (p_high - p_low + 1e-6), 0, 1)

nombres = ['Blue (450 nm)', 'Green (560 nm)', 'Red (650 nm)', 'Red Edge (730 nm)', 'NIR (840 nm)']
bandas_raw = [blue, green, red, re, nir]
colormaps = ['Blues', 'Greens', 'Reds', 'Oranges', 'YlOrRd']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, banda, nombre, cmap in zip(axes, bandas_raw, nombres, colormaps):
    im = ax.imshow(normalizar(banda), cmap=cmap, aspect='auto')
    ax.set_title(nombre, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Ortomosaico DJI P4M — 5 bandas espectrales\n(normalizado para visualización)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Resolución espacial: {blue.shape[1]} × {blue.shape[0]} px')

In [ ]:
# Composición color natural (R-G-B) y falso color NIR (NIR-R-G)
rgb = np.dstack([normalizar(red), normalizar(green), normalizar(blue)])
cir = np.dstack([normalizar(nir), normalizar(red), normalizar(green)])  # Color InfraRojo

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(rgb, aspect='auto')
ax1.set_title('Color Natural (R-G-B)', fontsize=12)
ax1.axis('off')

ax2.imshow(cir, aspect='auto')
ax2.set_title('Falso Color NIR (NIR-R-G)\nVegetación = magenta/rojo', fontsize=12)
ax2.axis('off')

plt.suptitle('Composiciones del ortomosaico P4M — Universidad del Magdalena', fontsize=12)
plt.tight_layout()
plt.show()

## Parte 3 — Calcular índices espectrales

In [ ]:
eps = 1e-6  # evitar división por cero

ndvi = (nir - red)  / (nir + red  + eps)
ndre = (nir - re)   / (nir + re   + eps)  # Red Edge: mejor que NDVI en vegetación densa
ndmi = (nir - blue) / (nir + blue + eps)  # Aproximación NDMI con bandas disponibles
savi = 1.5 * (nir - red) / (nir + red + 0.5 + eps)

# Histogramas para entender la distribución
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
datos_indices = [(ndvi,'NDVI','#27ae60'), (ndre,'NDRE','#e67e22'),
                 (ndmi,'NDMI','#3498db'), (savi,'SAVI','#9b59b6')]

for ax, (idx, nombre, color) in zip(axes.flat, datos_indices):
    valores = idx[(idx > -1) & (idx < 1)].ravel()
    ax.hist(valores, bins=80, color=color, alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.axvline(np.nanmean(valores), color='black', linestyle='--', linewidth=1.5,
               label=f'Media: {np.nanmean(valores):.3f}')
    ax.set_title(f'{nombre}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Valor del índice')
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Distribución de índices espectrales — Ortomosaico P4M', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización de los 4 índices en el mapa
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    (ndvi, 'NDVI', 'RdYlGn', (-0.2, 0.9)),
    (ndre, 'NDRE (Red Edge)', 'PiYG',   (0.0, 0.8)),
    (ndmi, 'NDMI aprox.', 'BrBG',   (-0.3, 0.7)),
    (savi, 'SAVI', 'RdYlGn', (-0.1, 0.7)),
]

for ax, (idx, titulo, cmap, (vmin, vmax)) in zip(axes.flat, configs):
    im = ax.imshow(np.clip(idx, vmin, vmax), cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.axis('off')
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Valor')

plt.suptitle('Índices espectrales — Ortomosaico DJI P4M\nUniversidad del Magdalena',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Parte 4 — Estadísticas por cobertura

In [ ]:
# Separar coberturas usando umbrales de NDVI y NDMI
# (esto lo afinaremos en el Notebook 08 con RF)
mascara_agua       = ndvi < 0.05
mascara_suelo      = (ndvi >= 0.05) & (ndvi < 0.25)
mascara_cesped     = (ndvi >= 0.25) & (ndvi < 0.50)
mascara_vegetacion = ndvi >= 0.50

coberturas = {
    'Agua / suelo húmedo': mascara_agua,
    'Suelo desnudo':        mascara_suelo,
    'Césped / pastizal':    mascara_cesped,
    'Vegetación densa':     mascara_vegetacion,
}

print(f"{'Cobertura':<25} {'Píxeles':>8} {'%':>6} {'NDVI':>6} {'NDRE':>6} {'NDMI':>6}")
print('-' * 60)
total_px = ndvi.size
for nombre, mascara in coberturas.items():
    n = np.sum(mascara)
    pct = 100 * n / total_px
    ndvi_m = np.nanmean(ndvi[mascara]) if n > 0 else 0
    ndre_m = np.nanmean(ndre[mascara]) if n > 0 else 0
    ndmi_m = np.nanmean(ndmi[mascara]) if n > 0 else 0
    print(f'{nombre:<25} {n:>8,} {pct:>6.1f}% {ndvi_m:>6.3f} {ndre_m:>6.3f} {ndmi_m:>6.3f}')

## Parte 5 — Exportar resultados como GeoTIFF

In [ ]:
def exportar_indice_geotiff(array, nombre_archivo, perfil_referencia, transform, crs):
    perfil_out = perfil_referencia.copy()
    perfil_out.update({'count': 1, 'dtype': 'float32', 'nodata': -9999})

    data = array.astype(np.float32)
    data[np.isnan(data)] = -9999

    with rasterio.open(nombre_archivo, 'w', **perfil_out) as dst:
        dst.write(data, 1)
    print(f'Exportado: {nombre_archivo}')

# Exportar los 4 índices
exportar_indice_geotiff(ndvi, 'ndvi_dron_universidad.tif',   perfil, transform, crs)
exportar_indice_geotiff(ndre, 'ndre_dron_universidad.tif',   perfil, transform, crs)
exportar_indice_geotiff(ndmi, 'ndmi_dron_universidad.tif',   perfil, transform, crs)
exportar_indice_geotiff(savi, 'savi_dron_universidad.tif',   perfil, transform, crs)

print()
print('Abre estos archivos en QGIS para ver los índices georreferenciados.')
print('En QGIS: Layer → Add Raster Layer → selecciona el .tif')
print('Asigna paleta: Layer Properties → Symbology → SingleBand Pseudocolor → RdYlGn')

---
## Resumen
Hoy aprendiste el flujo completo **dron → ortomosaico → análisis Python**:
1. `rasterio.open()` para leer GeoTIFF con metadatos espaciales
2. `src.read(banda)` devuelve un array NumPy
3. Los índices espectrales son operaciones matemáticas sobre arrays
4. `rasterio.open(..., 'w')` para exportar resultados georreferenciados

En el **Notebook 08** tomaremos estas 5 bandas y construiremos un clasificador Random Forest para mapear automáticamente el lago, el césped, los árboles y las áreas de concreto.